# Pendulum swing-up — value iteration vs LQR

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/learn/teaching/pendulum_swing_up_vi_vs_lqr.ipynb)

This notebook compares two ways to synthesize a policy for the **same** swing-up problem and the **same** quadratic cost $J$. Both return a state-feedback law $u=\pi(x)$; they differ in what they assume about the dynamics.

1. **Value iteration (VI)**: global dynamic programming on a grid — the discretized nonlinear optimum, with torque limits.
2. **LQR**: local linear-quadratic feedback from the linearized dynamics at the upright equilibrium.

Same plant, same $Q$ and $R$. The lesson is local vs global synthesis: LQR is cheap and exact near $\bar x$; VI can swing up from the hanging position.

This page uses the [minilink](https://github.com/alx87grd/minilink) toolbox. For the library workflow see [`showcase_minilink`](../intro/showcase_minilink.ipynb); LQR and planning are in [`03_control`](../intro/03_control.ipynb) and [`09_planning`](../intro/09_planning.ipynb).


In [ ]:
# Local: minilink already installed. Colab: clone + path.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from minilink.core.diagram import DiagramSystem
from minilink.core.trajectory import Trajectory
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum
from minilink.planning.policy_synthesis.discretizer import StateSpaceGrid
from minilink.planning.policy_synthesis.dp import (
    DynamicProgrammingOptions,
    DynamicProgrammingPlanner,
)
from minilink.planning.problems import PlanningProblem


## 1. Plant

We load a minilink catalog class (`Pendulum`) that already defines the equations of motion and the state/input labels. The state is $x = [\theta,\;\dot\theta]$ and the input is the pivot torque $u$. The dynamics are
$$\dot x = f(x,u).$$
Hanging down is $\theta = 0$; the upright target is $\bar x = [-\pi,\; 0]$. We also set the **domain** — bounds on $x$ and $|u|\le u_{\max}$ — used later by the grid and by LQR saturation.


In [ ]:
UPRIGHT = np.array([-np.pi, 0.0])  # target (upright) and LQR linearization point
X0 = np.array([0.0, 0.0])  # hanging down
TORQUE = 5.0
DT = 0.05
TF = 10.0
X_GRID = (201, 201)
U_GRID = (21,)
TOL = 0.1
INF = 500.0
Q = np.diag([1.0, 1.0])
R = np.diag([1.0])


def make_pendulum():
    plant = Pendulum()
    plant.state.lower_bound = np.array([-2.0 * np.pi, -2.0 * np.pi])
    plant.state.upper_bound = np.array([+2.0 * np.pi, +2.0 * np.pi])
    plant.inputs["u"].lower_bound = np.array([-TORQUE])
    plant.inputs["u"].upper_bound = np.array([+TORQUE])
    plant.x0 = X0.copy()
    return plant


plant = make_pendulum()


## 2. Cost function

Both controllers minimize the same quadratic performance metric
$$J = \int_0^{t_f} \big( (x-\bar x)' Q (x-\bar x) + u' R u \big)\, dt$$
with $Q = I$ and $R = I$. This $J$ is what we compare later on the closed-loop trajectories.


In [ ]:
from minilink.core.costs import QuadraticCost

cost = QuadraticCost.from_system(plant, xbar=UPRIGHT, Q=Q, R=R)

print("Target:", UPRIGHT)
print("Q=\n", cost.Q)
print("R=\n", cost.R)


## 3. Planning problem

A `PlanningProblem` packages the plant, the cost, and the goal. The optimal-control problem is
$$\min_{\pi}\; J \quad\text{s.t.}\quad \dot x = f\big(x,\pi(x)\big),\quad |u|\le u_{\max}.$$
Value iteration solves this on a grid. LQR solves a local linear-quadratic approximation of the same $J$.


In [ ]:
problem = PlanningProblem(plant, x_goal=UPRIGHT, cost=cost)


## 4. Value iteration

We discretize $x$ and $u$ on a grid and solve the discrete Bellman equation for the cost-to-go $J^*$:
$$J^*(x) = \min_u \Big\{ g(x,u)\,\Delta t + J^*\big(x + f(x,u)\,\Delta t\big) \Big\}.$$
The minimizing $u$ is the global (discretized) policy $\pi^*(x)$. Here the state grid is $201\times 201$, the torque has 21 levels, and $\Delta t = 0.05\,\mathrm{s}$.


In [ ]:
grid = StateSpaceGrid(problem, x_grid_shape=X_GRID, u_grid_shape=U_GRID, dt=DT)

planner = DynamicProgrammingPlanner(
    problem,
    grid=grid,
    options=DynamicProgrammingOptions(
        alpha=1.0,
        tol=TOL,
        max_iterations=2000,
        out_of_bound_cost=INF,
        verbose=True,
    ),
)

planner.solve()
planner.clean_infeasible_set()
vi_ctl = planner.get_controller()


## 5. LQR

Linearize the plant at the upright equilibrium $(\bar x, \bar u)$:
$$\dot{\tilde x} = A\tilde x + B\tilde u,\qquad \tilde x = x-\bar x.$$
The infinite-horizon LQR gain $K$ minimizes the same quadratic $J$ for this linear model, giving the local law
$$u = \bar u - K(x-\bar x).$$
Same $Q$, $R$, and plant as value iteration — but no torque limits in the synthesis, and no validity away from $\bar x$.


In [ ]:
from minilink.control.lqr import lqr_at_operating_point

lqr_ctl = lqr_at_operating_point(make_pendulum(), UPRIGHT, Q, R)
K = lqr_ctl.params["K"]
print("LQR gain K =", np.round(K, 3))


## 6. Control laws

Both maps $u=\pi(\theta,\dot\theta)$ on the same state box. LQR is a linear plane; VI is a nonlinear lookup that follows the natural dynamics. Near $\bar x$ they look similar; globally LQR asks for much larger torques.


In [ ]:
planner.plot_policy()
lqr_ctl.plot_control_law(
    bounds=((grid.x_lb[0], grid.x_ub[0]), (grid.x_lb[1], grid.x_ub[1])),
    vmin=-TORQUE,
    vmax=TORQUE,
    title="LQR control law",
)
vi_ctl.plot_control_law(title="VI control law (interpolated)")


## 7. Closed-loop simulation

We wire each policy as state feedback $u=\pi(x)$ and integrate from the hanging position $x_0 = [0,\; 0]$. LQR goes straight to the goal with large torque; VI pumps energy, then swings up.


In [ ]:
def closed_loop(controller, x0, name):
    """Wire a state-feedback controller to a fresh copy of the pendulum."""
    plant = make_pendulum()
    plant.x0 = np.array(x0)
    diagram = DiagramSystem()
    diagram.add_subsystem(controller, "ctl")
    diagram.add_subsystem(plant, "plant")
    diagram.connect("plant", "y", "ctl", "x")
    diagram.connect("ctl", "u", "plant", "u")
    diagram.name = name
    diagram.camera_scale = 2.0
    n_steps = int(TF / DT) + 1  # same step as the DP discretization
    traj = diagram.compute_trajectory(tf=TF, n_steps=n_steps, solver="euler")
    return diagram, plant, traj


def applied_u(controller, traj):
    """Reconstruct u(t) from a controller that implements action(x)."""
    return np.array([controller.action(x) for x in traj.x.T]).T

cl_vi, plant_vi, traj_vi = closed_loop(vi_ctl, X0, "Pendulum with VI")
cl_lqr, plant_lqr, traj_lqr = closed_loop(lqr_ctl, X0, "Pendulum with LQR")

cl_vi.plot_trajectory(traj_vi)
cl_lqr.plot_trajectory(traj_lqr)


## 8. Animation — VI

Closed-loop motion under the value-iteration policy, from hanging down.


In [ ]:
cl_vi.animate(traj_vi)


## 8. Animation — LQR

Same initial state under LQR. Compare the torque and the path to the VI animation.


In [ ]:
cl_lqr.animate(traj_lqr)


## 9. Phase plane

Trajectories in the $(\theta,\dot\theta)$ plane. The vector field is the unactuated dynamics $\dot x = f(x,0)$. VI rides those orbits; LQR cuts across them.


In [ ]:
plant_vi.plot_phase_plane(traj_vi)
plant_lqr.plot_phase_plane(traj_lqr)


## 10. Performance

The same $J$ evaluated along each closed-loop trajectory. VI is the global optimum of the discretized problem, so its $J$ should be lower from the hanging position. LQR typically spends more torque (and more cost) getting there.


In [ ]:
K_row = lqr_ctl.params["K"][0]
ubar = lqr_ctl.params["ubar"][0]
u_lqr = np.clip(
    ubar - (traj_lqr.x.T - UPRIGHT) @ K_row, -TORQUE, TORQUE
).reshape(1, -1)
traj_vi_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_vi.t, x=traj_vi.x, u=applied_u(vi_ctl, traj_vi))
)
traj_lqr_cost = cost.evaluate_trajectory(
    Trajectory(t=traj_lqr.t, x=traj_lqr.x, u=u_lqr)
)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(traj_vi_cost.t, traj_vi_cost.signals["cost"][0], label="VI")
ax.plot(traj_lqr_cost.t, traj_lqr_cost.signals["cost"][0], label="LQR")
ax.set_xlabel("t [s]")
ax.set_ylabel("$J$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()

